In [ ]:
import sys
print("Path agregado")

In [ ]:
import os  
import pandas as pd
import ee
import geemap
from datetime import datetime, timedelta
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely import union_all

# Autenticación e inicialización de Earth Engine
ee.Authenticate()
ee.Initialize(project='TU-PROYECTO-GEE') #nombre del proyecto de GEE

# Configuración
satelite = 'LANDSAT/LT05/C02/T1_L2'  # Landsat 5 Collection 2, Nivel 2
interDate = '2009-12-31'  # Ajusta esta fecha (Landsat 5 terminó en 2011) - 1984 - 2011 
value_cl = 30  # Porcentaje máximo de nubes permitido
shapefile_path = r"C:\ruta\a\tu\area_de_interes.shp"
buffer_distance = 5000  # Buffer en metros

def shapefile_to_ee_polygon(shapefile_path, buffer_distance=1000):
    try:
        gdf = gpd.read_file(shapefile_path)
        if gdf.crs is None:
            raise ValueError("El shapefile no tiene un sistema de coordenadas definido.")
        gdf = gdf.to_crs('EPSG:32719') 
        gdf['geometry'] = gdf.geometry.buffer(buffer_distance)
        gdf = gdf.to_crs('EPSG:4326')
        polygon = union_all(gdf.geometry)
        if isinstance(polygon, Polygon):
            coords = list(polygon.exterior.coords)
            return ee.Geometry.Polygon([coords])
        elif isinstance(polygon, MultiPolygon):
            multipolygon_coords = []
            for part in polygon.geoms:
                multipolygon_coords.append([list(part.exterior.coords)])
            return ee.Geometry.MultiPolygon(multipolygon_coords)
        else:
            raise ValueError("La geometría resultante no es válida.")
    except Exception as e:
        print(f"Error inesperado: {e}")
        return None

AOI = shapefile_to_ee_polygon(shapefile_path, buffer_distance)

if AOI:
    interDate = datetime.strptime(interDate, '%Y-%m-%d')
    startDate = interDate - timedelta(days=365)   #Cambiar de 1 día a 365 días (anual).
    endDate = interDate + timedelta(days=1)  #dejarlo por defecto (en todo caso jugar con este parametro) 
    print(f"Buscando imágenes entre {startDate} y {endDate}...")

    # Filtrar colección Landsat 5
    Collection_L5 = (
        ee.ImageCollection(satelite)
        .filterDate(startDate.strftime('%Y-%m-%d'), endDate.strftime('%Y-%m-%d'))
        .filterBounds(AOI)
        .filter(ee.Filter.lt('CLOUD_COVER_LAND', value_cl))
    )

    n = Collection_L5.size().getInfo()
    if n == 0:
        raise ValueError("No se encontraron imágenes Landsat 5 para ese período.")
    else:
        print(f'Total de imágenes encontradas: {n}')
        image_info = Collection_L5.aggregate_array('system:time_start').getInfo()
        cloud_coverage = Collection_L5.aggregate_array('CLOUD_COVER_LAND').getInfo()

        df = pd.DataFrame({
            'Fecha': [datetime.utcfromtimestamp(date / 1000).strftime('%Y-%m-%d') for date in image_info],
            'Porcentaje': cloud_coverage
        })

        df_agrupado = df.groupby('Fecha', as_index=False)['Porcentaje'].mean()
        print(df_agrupado)

        # Crear el mapa
        Map = geemap.Map()

        for fecha in df_agrupado['Fecha']:
            next_day = (datetime.strptime(fecha, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
            Collection_fecha = (
                ee.ImageCollection(satelite)
                .filterDate(fecha, next_day)
                .filterBounds(AOI)
            )

            # Calcular mediana y aplicar escala
            medianpixels = Collection_fecha.median().clip(AOI).multiply(0.0000275).add(-0.2)

            # Reducir para obtener valores estadísticos
            stats = medianpixels.reduceRegion(
                reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
                geometry=AOI,
                scale=30,
                maxPixels=1e13
            ).getInfo()

            try:
                vis_params = {
                    'bands': ['SR_B5', 'SR_B4', 'SR_B3'],
                    'min': min(stats[f'{band}_mean'] - stats[f'{band}_stdDev'] for band in ['SR_B5', 'SR_B4', 'SR_B3']),
                    'max': max(stats[f'{band}_mean'] + stats[f'{band}_stdDev'] for band in ['SR_B5', 'SR_B4', 'SR_B3']),
                    'gamma': 1.8
                }
                #Analizis de bandas 

                #NDVI: SWIR-RED/SWIR+RED
                #Se podría realizar geoprocesos 

                Map.addLayer(medianpixels, vis_params, f"Landsat5_{fecha}")
                print(f"Agregada imagen de {fecha} al mapa.")
            except Exception as e:
                print(f"No se pudo visualizar la imagen de {fecha}: {e}")

        Map
        print("Visualización de todas las fechas completada.")






